In [1]:
########### Python 3.2 #############
import http.client, urllib.request, urllib.parse, urllib.error, base64
import json
import random 
import requests
import time
import uuid
import datetime
import pytz
from datetime import datetime, timedelta
import os
import sys

In [2]:
run_time = datetime.now()
run_time = pytz.timezone("America/New_York").localize(run_time)

In [3]:
personalization_base_url = "https://bwh-pharmacoepi-roybal-dev-use2-cog.cognitiveservices.azure.com/"
with open("../Scripts/.keys/azure-personalizer-key.txt", 'r') as f:
     resource_key = f.read().rstrip()
personalization_model_properties_url = personalization_base_url + "personalizer/v1.0/model/properties"
personalization_model_policy_url = personalization_base_url + "personalizer/v1.0/configurations/policy"
personalization_service_configuration_url = personalization_base_url + "personalizer/v1.0/configurations/service"
headers = {'Ocp-Apim-Subscription-Key' : resource_key, 'Content-Type': 'application/json'}

In [4]:
def get_last_updated():
    
    print('-----checking model')
    
    # get model properties
    response = requests.get(personalization_model_properties_url, headers = headers, params = None)
    
    print(response)
    print(response.json())
    
    # get lastModifiedTime
    lastModifiedTime = json.dumps(response.json()["lastModifiedTime"])
    
    print(f'-----model updated: {lastModifiedTime}')
    settings = response.json()
    return settings

In [5]:
def get_service_settings():
    
    print('-----checking service settings')
    
    # get learning policy
    response = requests.get(personalization_model_policy_url, headers = headers, params = None)
    
    print(response)
    print(response.json())
    props1 = response.json()
    
    # get service settings
    response = requests.get(personalization_service_configuration_url, headers = headers, params = None)
    
    print(response)
    print(response.json())
    props2 = response.json()
    
    return props1, props2

In [8]:
# ########### Python 3.2 #############
# import http.client, urllib.request, urllib.parse, urllib.error, base64

# headers = {
#     # Request headers
#     'Ocp-Apim-Subscription-Key': resource_key,
# }

# params = urllib.parse.urlencode({
# })

# try:
#     conn = http.client.HTTPSConnection('eastus2.api.cognitive.microsoft.com')
#     conn.request("GET", "/personalizer/v1.0/model/properties?%s" % params, "{body}", headers)
#     response = conn.getresponse()
#     data = response.read()
#     print(data)
#     conn.close()
# except Exception as e:
#     print("[Errno {0}] {1}".format(e.errno, e.strerror))

# ####################################

In [6]:
#data

In [7]:
#props = json.loads(data.decode('utf-8'))

In [8]:
#props

In [9]:
#modelLastModified = props['lastModifiedTime']

In [10]:
settings = get_last_updated()
props1, props2 = get_service_settings()

-----checking model
<Response [200]>
{'creationTime': '2020-08-27T17:24:36+00:00', 'lastModifiedTime': '2021-01-14T12:43:53+00:00'}
-----model updated: "2021-01-14T12:43:53+00:00"
-----checking service settings
<Response [200]>
{'name': '97725378e38c42299eb60adec47570a7', 'arguments': '--cb_explore_adf --epsilon 0.2 --power_t 0 -l 0.001 --cb_type mtr -q ::'}
<Response [200]>
{'rewardWaitTime': 'P2D', 'defaultReward': 0.0, 'rewardAggregation': 'earliest', 'explorationPercentage': 0.1, 'modelExportFrequency': 'PT12H', 'logRetentionDays': 9999, 'modelAutoPublish': True, 'stagedModelHistoryLength': 10, 'lastConfigurationEditDate': '2020-12-23T19:25:44', 'learningMode': 'Online'}


In [11]:
settings

{'creationTime': '2020-08-27T17:24:36+00:00',
 'lastModifiedTime': '2021-01-14T12:43:53+00:00'}

In [12]:
props1

{'name': '97725378e38c42299eb60adec47570a7',
 'arguments': '--cb_explore_adf --epsilon 0.2 --power_t 0 -l 0.001 --cb_type mtr -q ::'}

In [13]:
props2

{'rewardWaitTime': 'P2D',
 'defaultReward': 0.0,
 'rewardAggregation': 'earliest',
 'explorationPercentage': 0.1,
 'modelExportFrequency': 'PT12H',
 'logRetentionDays': 9999,
 'modelAutoPublish': True,
 'stagedModelHistoryLength': 10,
 'lastConfigurationEditDate': '2020-12-23T19:25:44',
 'learningMode': 'Online'}

In [14]:
props1['arguments']

'--cb_explore_adf --epsilon 0.2 --power_t 0 -l 0.001 --cb_type mtr -q ::'

In [15]:
name = 'eval_'+str(run_time.date())
name

'eval_2021-01-14'

In [17]:
endTime = str(run_time.date()) + 'T00:00:00Z'
endTime

'2021-01-14T00:00:00Z'

In [18]:
body = {"enableOfflineExperimentation": 'true',
  "name": name,
  "startTime": "2020-12-24T00:00:00Z",
  "endTime": endTime,
  "policies": [
   props1
  ]
}

In [19]:
headers = {
    # Request headers
    'Content-Type': 'application/json-patch+json',
    'Ocp-Apim-Subscription-Key': resource_key,
}

params = urllib.parse.urlencode({
})

try:
    conn = http.client.HTTPSConnection('eastus2.api.cognitive.microsoft.com')
    conn.request("POST", "/personalizer/v1.0/evaluations?%s" % params, str(body), headers)
    response = conn.getresponse()
    data = response.read()
    evaluation = json.loads(data.decode('utf-8'))
    print(data)
    conn.close()
except Exception as e:
    print("[Errno {0}] {1}".format(e.errno, e.strerror))

b'{\n  "id": "22cd7068-cf0e-4173-96d9-0a513b8ffa94",\n  "name": "eval_2021-01-14",\n  "startTime": "2020-12-24T00:00:00Z",\n  "endTime": "2021-01-14T00:00:00Z",\n  "jobId": "22cd7068-cf0e-4173-96d9-0a513b8ffa94",\n  "status": "pending",\n  "policyResults": [\n    {\n      "name": "97725378e38c42299eb60adec47570a7",\n      "arguments": "--cb_explore_adf --epsilon 0.2 --power_t 0 -l 0.001 --cb_type mtr -q :: --dsjson"\n    }\n  ],\n  "featureImportance": []\n}'


In [24]:
evaluationId = evaluation['id']

In [25]:
evaluationId

'dd16ad33-cb33-4454-8dd5-eb973b3732ce'

In [28]:
headers = {
    # Request headers
    'Ocp-Apim-Subscription-Key': resource_key,
}

params = urllib.parse.urlencode({
})

try:
    conn = http.client.HTTPSConnection('eastus2.api.cognitive.microsoft.com')
    conn.request("GET", "/personalizer/v1.0/evaluations/"+evaluationId+"?%s" % params, "{body}", headers)
    response = conn.getresponse()
    data = response.read()
    evaluation_status = json.loads(data.decode('utf-8'))
    #print(data)
    conn.close()
except Exception as e:
    print("[Errno {0}] {1}".format(e.errno, e.strerror))

####################################

In [29]:
evaluation_status

{'id': 'dd16ad33-cb33-4454-8dd5-eb973b3732ce',
 'name': 'eval_2021-01-13',
 'startTime': '2020-12-24T00:00:00Z',
 'endTime': '2021-01-12T00:00:00Z',
 'jobId': 'dd16ad33-cb33-4454-8dd5-eb973b3732ce',
 'status': 'completed',
 'policyResults': [{'name': '97725378e38c42299eb60adec47570a7',
   'arguments': '--cb_explore_adf --epsilon 0.2 --power_t 0 -l 0.001 --cb_type mtr -q :: --dsjson',
   'summary': [{'timeStamp': '2020-12-24T13:35:00Z',
     'ipsEstimatorNumerator': 5.3333325,
     'ipsEstimatorDenominator': 16.0,
     'snipsEstimatorDenominator': 14.4,
     'aggregateTimeWindow': 'PT5M',
     'nonZeroProbability': 16.0,
     'confidenceInterval': 0.9,
     'sumOfSquares': 4.417776},
    {'timeStamp': '2020-12-24T13:40:00Z',
     'ipsEstimatorNumerator': 0.0,
     'ipsEstimatorDenominator': 0.0,
     'snipsEstimatorDenominator': 0.0,
     'aggregateTimeWindow': 'PT5M',
     'nonZeroProbability': 0.0,
     'confidenceInterval': 0.0,
     'sumOfSquares': 0.0},
    {'timeStamp': '2020-12-2

In [181]:
fp = 'evaluation_log_'+str(run_time.date())+'.txt'
old_stdout = sys.stdout        
log_file = open(fp, "w")
sys.stdout = log_file
print(evaluation_status)
print("-----------------------------------------------------------------------------------")
log_file.close()
sys.stdout = old_stdout

In [188]:
sys.stdout = old_stdout

In [32]:
print(evaluation_status['policyResults'][0]['summary'])

[{'timeStamp': '2020-12-24T13:35:00Z', 'ipsEstimatorNumerator': 5.3333325, 'ipsEstimatorDenominator': 16.0, 'snipsEstimatorDenominator': 14.4, 'aggregateTimeWindow': 'PT5M', 'nonZeroProbability': 16.0, 'confidenceInterval': 0.9, 'sumOfSquares': 4.417776}, {'timeStamp': '2020-12-24T13:40:00Z', 'ipsEstimatorNumerator': 0.0, 'ipsEstimatorDenominator': 0.0, 'snipsEstimatorDenominator': 0.0, 'aggregateTimeWindow': 'PT5M', 'nonZeroProbability': 0.0, 'confidenceInterval': 0.0, 'sumOfSquares': 0.0}, {'timeStamp': '2020-12-24T13:45:00Z', 'ipsEstimatorNumerator': 0.0, 'ipsEstimatorDenominator': 0.0, 'snipsEstimatorDenominator': 0.0, 'aggregateTimeWindow': 'PT5M', 'nonZeroProbability': 0.0, 'confidenceInterval': 0.0, 'sumOfSquares': 0.0}, {'timeStamp': '2020-12-24T13:50:00Z', 'ipsEstimatorNumerator': 0.0, 'ipsEstimatorDenominator': 0.0, 'snipsEstimatorDenominator': 0.0, 'aggregateTimeWindow': 'PT5M', 'nonZeroProbability': 0.0, 'confidenceInterval': 0.0, 'sumOfSquares': 0.0}, {'timeStamp': '2020-1

In [33]:
print(evaluation_status['policyResults'][0]['totalSummary'])

{'timeStamp': '2021-01-12T00:00:00Z', 'ipsEstimatorNumerator': 186.43109, 'ipsEstimatorDenominator': 2636.0, 'snipsEstimatorDenominator': 2492.0422, 'aggregateTimeWindow': 'PT0S', 'nonZeroProbability': 2636.0, 'confidenceInterval': 2.1666675, 'sumOfSquares': 134.30336}


In [36]:
evaluation_status['featureImportance']

[['Context.response_action_id_contentyesContent with observed_feedback_features.avg_adherence_1day',
  'Context.response_action_id_framingnegFrame',
  'Context.response_action_id_framingnegFrame with Context.response_action_id_framingnegFrame',
  'Context.response_action_id_framingnegFrame with Context.response_action_id_socialnoSocial',
  'Context.response_action_id_framingnegFrame with clinical_features.hba1c8.1-8.9',
  'Context.response_action_id_framingnegFrame with clinical_features.hba1c9.0-9.9',
  'Context.response_action_id_framingnegFrame with clinical_features.num_physicians4+',
  'Context.response_action_id_framingnegFrame with clinical_features.num_years_dm_rx1-2',
  'Context.response_action_id_framingnegFrame with demographic_features.age45-54',
  'Context.response_action_id_framingnegFrame with demographic_features.education_levelCollege_grad/Postgrad',
  'Context.response_action_id_framingnegFrame with demographic_features.education_levelHS_or_below/HS_grad',
  'Context.

In [35]:
evaluation_status

{'id': 'dd16ad33-cb33-4454-8dd5-eb973b3732ce',
 'name': 'eval_2021-01-13',
 'startTime': '2020-12-24T00:00:00Z',
 'endTime': '2021-01-12T00:00:00Z',
 'jobId': 'dd16ad33-cb33-4454-8dd5-eb973b3732ce',
 'status': 'completed',
 'policyResults': [{'name': '97725378e38c42299eb60adec47570a7',
   'arguments': '--cb_explore_adf --epsilon 0.2 --power_t 0 -l 0.001 --cb_type mtr -q :: --dsjson',
   'summary': [{'timeStamp': '2020-12-24T13:35:00Z',
     'ipsEstimatorNumerator': 5.3333325,
     'ipsEstimatorDenominator': 16.0,
     'snipsEstimatorDenominator': 14.4,
     'aggregateTimeWindow': 'PT5M',
     'nonZeroProbability': 16.0,
     'confidenceInterval': 0.9,
     'sumOfSquares': 4.417776},
    {'timeStamp': '2020-12-24T13:40:00Z',
     'ipsEstimatorNumerator': 0.0,
     'ipsEstimatorDenominator': 0.0,
     'snipsEstimatorDenominator': 0.0,
     'aggregateTimeWindow': 'PT5M',
     'nonZeroProbability': 0.0,
     'confidenceInterval': 0.0,
     'sumOfSquares': 0.0},
    {'timeStamp': '2020-12-2

In [44]:
fp = 'full_evaluation_log_'+str(run_time.date())+'.txt'
old_stdout = sys.stdout        
log_file = open(fp, "w")
sys.stdout = log_file
print(evaluation_status)
print("-----------------------------------------------------------------------------------")
log_file.close()
sys.stdout = old_stdout

In [42]:
fp = 'evaluation_log_'+str(run_time.date())+'.txt'
old_stdout = sys.stdout        
log_file = open(fp, "w")
sys.stdout = log_file
print(evaluation_status['policyResults'][0]['totalSummary'])
print("-----------------------------------------------------------------------------------")
log_file.close()
sys.stdout = old_stdout


fp = 'feature_importance_log_'+str(run_time.date())+'.txt'
old_stdout = sys.stdout        
log_file = open(fp, "w")
sys.stdout = log_file
print(evaluation_status['featureImportance'])
print("-----------------------------------------------------------------------------------")
log_file.close()
sys.stdout = old_stdout

In [39]:
evaluation_status['featureImportance']

[['Context.response_action_id_contentyesContent with observed_feedback_features.avg_adherence_1day',
  'Context.response_action_id_framingnegFrame',
  'Context.response_action_id_framingnegFrame with Context.response_action_id_framingnegFrame',
  'Context.response_action_id_framingnegFrame with Context.response_action_id_socialnoSocial',
  'Context.response_action_id_framingnegFrame with clinical_features.hba1c8.1-8.9',
  'Context.response_action_id_framingnegFrame with clinical_features.hba1c9.0-9.9',
  'Context.response_action_id_framingnegFrame with clinical_features.num_physicians4+',
  'Context.response_action_id_framingnegFrame with clinical_features.num_years_dm_rx1-2',
  'Context.response_action_id_framingnegFrame with demographic_features.age45-54',
  'Context.response_action_id_framingnegFrame with demographic_features.education_levelCollege_grad/Postgrad',
  'Context.response_action_id_framingnegFrame with demographic_features.education_levelHS_or_below/HS_grad',
  'Context.

In [20]:
try:
    conn = http.client.HTTPSConnection('eastus2.api.cognitive.microsoft.com')
    conn.request("GET", "/personalizer/v1.0/model?%s" % params, "{body}", headers)
    response = conn.getresponse()
    data = response.read()
    #print(data)
    conn.close()
except Exception as e:
    print("[Errno {0}] {1}".format(e.errno, e.strerror))

In [22]:
#data

In [40]:
# json.dumps(response.json())

In [23]:
from vowpalwabbit import pyvw

ModuleNotFoundError: No module named 'vowpalwabbit'

## Resources:
* https://github.com/Azure-Samples/cognitive-services-personalizer-samples/blob/master/samples/azurenotebook/Personalizer.ipynb
* https://westus2.dev.cognitive.microsoft.com/docs/services/personalizer-api/operations/CreateEvaluation